# 1. Introduction - Multi-Agentic Workflow

This notebook demonstrates a practical **multi-agentic workflow** for campaign creation. Instead of asking one model to do everything, we split the work into specialized agents that collaborate through structured handoffs.

### Why this pattern works
- Better specialization: each agent focuses on one job (research, strategy, creative, copy, packaging).
- More traceability: each output is inspectable and easier to validate.
- Easier iteration: you can replace or tune one agent without rewriting the full pipeline.

## Use Case - Bank Marketing Campaign Generation

Goal: generate a data-informed marketing campaign brief for a bank term-deposit conversion use case.

The workflow turns historical campaign data into:
- Segment-aware strategy recommendations
- Visual direction for campaign assets
- Suggestions for channels
- A final packaged report for stakeholders
- A campaign graphic adapted to strategic segments

## Dataset Overview

We use `bank.csv` (UCI Bank Marketing style schema) as the grounding dataset.

Key points:
- Target variable: `y` (term deposit subscription, yes/no).
- Mixed features: demographic, contact, and campaign-history fields.
- The analysis computes simple evidence tables to support strategy and messaging decisions.

# 2. Setup & Environment

This section initializes imports, loads environment variables, and prepares model clients used by downstream agents.

In this step, we import the key libraries that support the agent workflow:

- **`openai`**: Used to interact with the OpenAI API for generating responses.
- **`utils`**: A custom helper module for this project. It includes utility functions for dataset handling, result formatting, and lightweight evaluation support.

The OpenAI key must be provided in a `.env` file.

The `bank.csv` file must be available in the **data** folder

In [1]:
# Setup: imports and clients
import sys
sys.path.append("../../agentic-ai")
import os
import re
import json
from datetime import datetime
from typing import Dict, Any
import pandas as pd
import openai
from utils import utils

from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

openai_client = None
if OPENAI_API_KEY:
    try:
        openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
    except Exception as e:
        print("OpenAI client init failed:", e)


print('Imports and clients initialized. Configure OPENAI_API_KEY to enable model calls.')


Imports and clients initialized. Configure OPENAI_API_KEY to enable model calls.


In [5]:
def load_bank_data(path: str = "bank.csv") -> pd.DataFrame:
    """Load the UCI Bank Marketing CSV into a pandas DataFrame.

    Args:
        path (str): Path to the CSV file (default 'bank.csv').

    Returns:
        pd.DataFrame: Loaded dataframe using ';' as the separator.
    """
    return pd.read_csv(path, sep=';')

df = load_bank_data("../data/bank.csv")
print("Shape:", df.shape)
display(df.head())
print("Baseline conversion rate:", (df["y"] == "yes").mean())


Shape: (4521, 17)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


Baseline conversion rate: 0.11523999115239991


## 3. Tools

Utility functions in this section handle safe parsing, data extraction, and reusable helpers shared across agents.

In [6]:
def safe_json_loads(text: str) -> Dict[str, Any]:
    """Safely parse JSON returned inside free-form text blocks.

    Args:
        text (str): The input text containing JSON, possibly with markdown formatting.

    Returns:
        Dict[str, Any]: The parsed JSON object.
    """
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return json.loads(text)


def call_llm_json(system_prompt: str, user_prompt: str, model: str = None) -> Dict[str, Any]:
    """Call the LLM via `openai_client` and parse a JSON response.

    Args:
        system_prompt (str): System-level prompt to set agent behavior.
        user_prompt (str): User-facing prompt describing the task.
        model (str): Model identifier to use for the call. Falls back to OPENAI_MODEL or gpt-4.1-mini.

    Returns:
        Dict[str, Any]: Parsed JSON response from the model.
    """
    if openai_client is None:
        raise RuntimeError("OPENAI_API_KEY is missing or OpenAI client is not initialized.")

    # Allow legacy provider-prefixed names like 'gpt-4.1-mini'.
    chosen_model = model or os.getenv("OPENAI_MODEL", "gpt-4.1-mini")
    if chosen_model.startswith("openai:"):
        chosen_model = chosen_model.split(":", 1)[1]

    resp = openai_client.chat.completions.create(
        model=chosen_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return safe_json_loads(resp.choices[0].message.content)


def compute_evidence_tables(df: pd.DataFrame) -> Dict[str, Any]:
    """Compute simple evidence tables used to ground research insights.

    Args:
        df (pd.DataFrame): Raw bank marketing dataframe.

    Returns:
        Dict[str, Any]: A dictionary containing overall conversion and sample tables.
    """
    dfx = df.copy()
    dfx["converted"] = (dfx["y"] == "yes").astype(int)
    overall = float(dfx["converted"].mean())

    contact = (
        dfx.groupby("contact", as_index=False)
        .agg(conversion_rate=("converted", "mean"), sample_size=("converted", "count"))
        .sort_values("conversion_rate", ascending=False)
    )
    month = (
        dfx.groupby("month", as_index=False)
        .agg(conversion_rate=("converted", "mean"), sample_size=("converted", "count"))
        .sort_values("conversion_rate", ascending=False)
    )
    job = (
        dfx.groupby("job", as_index=False)
        .agg(conversion_rate=("converted", "mean"), sample_size=("converted", "count"))
        .sort_values("conversion_rate", ascending=False)
    )

    return {
        "overall_conversion_rate": overall,
        "contact_table": contact.head(8).to_dict(orient="records"),
        "month_table": month.head(12).to_dict(orient="records"),
        "job_table": job.head(10).to_dict(orient="records"),
    }


def generate_image(prompt: str, out_dir: str = "outputs", size: str = "1024x1024") -> str:
    """Create a campaign visual or a placeholder describing the prompt.

    Args:
        prompt (str): Natural-language image prompt.
        out_dir (str): Directory where image or placeholder will be saved.
        size (str): Desired image size string for the image API.

    Returns:
        str: Path to the generated image file or placeholder text file.
    """
    os.makedirs(out_dir, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    image_path = os.path.join(out_dir, f"campaign_visual_{ts}.png")

    if openai_client is None:
        placeholder = os.path.join(out_dir, f"campaign_visual_{ts}.txt")
        with open(placeholder, "w", encoding="utf-8") as f:
            f.write("Image generation disabled. Prompt:\n\n" + prompt)
        return placeholder

    result = openai_client.images.generate(
        model="gpt-image-1",
        prompt=prompt,
        size=size,
    )
    import base64
    with open(image_path, "wb") as f:
        f.write(base64.b64decode(result.data[0].b64_json))
    return image_path


## 4. Agent Definitions

Each agent receives structured inputs and returns structured outputs for the next stage.

### 4.1 Research Agent

In [7]:
def research_agent(df: pd.DataFrame) -> Dict[str, Any]:
    """Analyze dataset evidence and return structured research insights."""
    evidence = compute_evidence_tables(df)
    system_prompt = "You are a Bank Marketing Research Agent. Return only valid JSON."
    user_prompt = f"""
Analyze the dataset evidence and return JSON with keys:
- top_segments
- best_channel
- best_months
- risks
- insight_summary
- evidence_tables

Evidence:
{json.dumps(evidence, ensure_ascii=True)}

Rules:
- Do not fabricate metrics.
- Keep insight_summary <= 120 words.
"""
    result = call_llm_json(system_prompt, user_prompt)
    if "evidence_tables" not in result:
        result["evidence_tables"] = evidence
    return result


### 4.2 Strategy Agent

In [8]:
def strategy_agent(research: Dict[str, Any]) -> Dict[str, Any]:
    """Create a practical 4-8 week marketing strategy from research JSON.

    Args:
        research (Dict[str, Any]): Research output produced by `research_agent`.

    Returns:
        Dict[str, Any]: Strategy JSON including campaign_goal, target_segments,
            channel_plan, timing_plan, offer_angle, kpi_targets, and assumptions.
    """
    system_prompt = "You are a Marketing Strategy Agent for bank campaigns. Return only valid JSON."
    user_prompt = f"""
Using this research JSON, create a practical 4-8 week strategy.

Research:
{json.dumps(research, ensure_ascii=True)}

Return keys:
- campaign_goal
- target_segments
- channel_plan
- timing_plan
- offer_angle
- kpi_targets
- assumptions
"""
    return call_llm_json(system_prompt, user_prompt)


### 4.3 Graphic Designer Agent

In [9]:
def graphic_designer_agent(strategy: Dict[str, Any], research: Dict[str, Any], size: str = "1024x1024") -> Dict[str, Any]:
    """Generate a visual concept for the campaign based on strategy and research.

    Args:
        strategy (Dict[str, Any]): Strategy JSON from `strategy_agent`.
        research (Dict[str, Any]): Research JSON from `research_agent`.
        size (str): Desired image size string for generation.

    Returns:
        Dict[str, Any]: Design JSON with `visual_direction`, `image_prompt`,
            `caption_options`, `compliance_notes`, and `image_path`.
    """
    system_prompt = "You are a Graphic Designer Agent for bank campaigns. Return only valid JSON."
    user_prompt = f"""
Create one campaign visual concept based on strategy and research.

Strategy:
{json.dumps(strategy, ensure_ascii=True)}

Research:
{json.dumps(research, ensure_ascii=True)}

Return keys:
- visual_direction
- image_prompt
- caption_options
- compliance_notes
"""
    design = call_llm_json(system_prompt, user_prompt)
    image_path = generate_image(design.get("image_prompt", ""), size=size)
    return {
        "visual_direction": design.get("visual_direction", ""),
        "image_prompt": design.get("image_prompt", ""),
        "caption_options": design.get("caption_options", []),
        "compliance_notes": design.get("compliance_notes", []),
        "image_path": image_path,
    }


### 4.4 Copywriter Agent

In [10]:
def copywriter_agent(strategy: Dict[str, Any], visual: Dict[str, Any], research: Dict[str, Any]) -> Dict[str, Any]:
    """Write campaign copy grounded in strategy, visual direction, and evidence.

    Args:
        strategy (Dict[str, Any]): Strategy JSON from `strategy_agent`.
        visual (Dict[str, Any]): Design JSON from `graphic_designer_agent`.
        research (Dict[str, Any]): Research JSON from `research_agent`.

    Returns:
        Dict[str, Any]: Copy JSON containing headline, body_copy, cta, channel_variants, and justification.
    """
    system_prompt = "You are a Copywriter Agent for bank campaigns. Return only valid JSON."
    visual_wo_path = {k: v for k, v in visual.items() if k != "image_path"}
    user_prompt = f"""
Write copy grounded in strategy, visual direction, and evidence.

Strategy:
{json.dumps(strategy, ensure_ascii=True)}

Visual:
{json.dumps(visual_wo_path, ensure_ascii=True)}

Research:
{json.dumps(research, ensure_ascii=True)}

Return keys:
- headline
- body_copy
- cta
- channel_variants
- justification
"""
    return call_llm_json(system_prompt, user_prompt)


### 4.5 Packaging Agent

In [11]:
def packaging_agent(
    research: Dict[str, Any],
    strategy: Dict[str, Any],
    visual: Dict[str, Any],
    copy: Dict[str, Any],
    output_path: str = None
) -> str:
    """Package research, strategy, creative, and copy into a markdown brief."""
    if output_path is None:
        output_path = f"bank_campaign_summary_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}.md"

    # Robust handling when model returns slightly different shapes
    best_channel_raw = research.get("best_channel", {}) if isinstance(research, dict) else {}
    if isinstance(best_channel_raw, dict):
        best_channel_text = best_channel_raw.get("channel", "N/A")
    else:
        best_channel_text = str(best_channel_raw) if best_channel_raw else "N/A"

    best_months_raw = research.get("best_months", []) if isinstance(research, dict) else []
    best_months_text = ", ".join(map(str, best_months_raw)) if isinstance(best_months_raw, list) else str(best_months_raw)

    risks_raw = research.get("risks", []) if isinstance(research, dict) else []
    risks_text = ", ".join(map(str, risks_raw)) if isinstance(risks_raw, list) else str(risks_raw)

    content = f"""# Bank Marketing Campaign Brief
## 1. Executive Summary
{research.get('insight_summary', '') if isinstance(research, dict) else ''}
## 2. Evidence
- Overall conversion rate: {(research.get('evidence_tables', {}) if isinstance(research, dict) else {}).get('overall_conversion_rate', 'N/A')}
- Best channel: {best_channel_text}
- Best months: {best_months_text}
- Risks: {risks_text}
## 3. Strategy
- Goal: {strategy.get('campaign_goal', '') if isinstance(strategy, dict) else ''}
- Offer angle: {strategy.get('offer_angle', '') if isinstance(strategy, dict) else ''}
### Target Segments
{json.dumps(strategy.get('target_segments', []) if isinstance(strategy, dict) else [], indent=2)}
### KPI Targets
{json.dumps(strategy.get('kpi_targets', []) if isinstance(strategy, dict) else [], indent=2)}
## 4. Creative
![Campaign Visual]({visual.get('image_path', '') if isinstance(visual, dict) else ''})
- Visual direction: {visual.get('visual_direction', '') if isinstance(visual, dict) else ''}
- Caption options: {json.dumps(visual.get('caption_options', []) if isinstance(visual, dict) else [], ensure_ascii=True)}
## 5. Copy
- Headline: {copy.get('headline', '') if isinstance(copy, dict) else ''}
- Body: {copy.get('body_copy', '') if isinstance(copy, dict) else ''}
- CTA: {copy.get('cta', '') if isinstance(copy, dict) else ''}
### Channel Variants
{json.dumps(copy.get('channel_variants', []) if isinstance(copy, dict) else [], indent=2)}
## 6. Justification
{copy.get('justification', '') if isinstance(copy, dict) else ''}
## 7. Evidence Tables
{json.dumps(research.get('evidence_tables', {}) if isinstance(research, dict) else {}, indent=2)}
"""

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(content)

    return output_path


## 5. Full Multi-Agent Pipeline

In [12]:
def validate_claim_support(research: Dict[str, Any], strategy: Dict[str, Any], copy: Dict[str, Any]) -> Dict[str, Any]:
    """Run lightweight checks to ensure campaign claims are supported by evidence."""
    issues = []
    if not research.get("evidence_tables"):
        issues.append("Missing evidence tables")
    if not strategy.get("target_segments"):
        issues.append("No target segments")
    if not copy.get("headline"):
        issues.append("Missing headline")
    return {"ok": len(issues) == 0, "issues": issues}

def run_bank_marketing_campaign_pipeline(data_path: str = "bank.csv", output_path: str = None) -> Dict[str, Any]:
    """
    Execute the full multi-agent bank marketing pipeline.
    """
    df = load_bank_data(data_path)

    research = research_agent(df)
    print("[1/5] Research complete")

    strategy = strategy_agent(research)
    print("[2/5] Strategy complete")

    visual = graphic_designer_agent(strategy, research)
    print("[3/5] Graphic design complete")

    copy = copywriter_agent(strategy, visual, research)
    print("[4/5] Copywriting complete")

    validation = validate_claim_support(research, strategy, copy)
    if not validation["ok"]:
        print("Validation warnings:", validation["issues"])

    report_path = packaging_agent(research, strategy, visual, copy, output_path)
    print("[5/5] Packaging complete")
    print("Report path:", report_path)

    return {
        "research": research,
        "strategy": strategy,
        "visual": visual,
        "copy": copy,
        "validation": validation,
        "report_path": report_path,
    }


### Run Workflow

In [14]:
# Ensure bank.csv exists in the current directory
results = run_bank_marketing_campaign_pipeline("../data/bank.csv")
results["report_path"]


[1/5] Research complete
[2/5] Strategy complete
[3/5] Graphic design complete
[4/5] Copywriting complete
[5/5] Packaging complete
Report path: bank_campaign_summary_2026-03-14_12-53-28.md


'bank_campaign_summary_2026-03-14_12-53-28.md'

### View Report

In [16]:
with open(results["report_path"], "r", encoding="utf-8") as f:
    utils.print_html(f.read())


## 10. Key Takeaways

What this notebook demonstrates in practice:

- **Multi-agent decomposition:** Decomposing a marketing workflow into focused agents (research, strategy, design, copy, packaging) makes responsibilities explicit and outputs auditable.

- **Data grounding:** Compute simple evidence tables from `bank.csv` to anchor strategic choices and justify creative/copy claims.

- **Practical outputs:** The pipeline produces a sharable campaign brief that combines quantitative evidence with creative direction and channel plans.

- **Validation matters:** Lightweight validation checks (e.g., ensuring evidence tables present) reduce the risk of unsupported claims in campaign materials.

- **Extensibility:** This pattern is extensible - add A/B testing agents, budget optimizers, or measurement hooks to move from brief to execution.

### Final thought

A layered, agentic approach lets teams iterate quickly while staying grounded in data - enabling creative experimentation without losing rigor.